# 🧪 Sesión 20 · Hito 2: API con Inyección de Dependencias, Auth JWT y RBAC

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿La seguridad de tu API resiste una auditoría?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/registro   {"usuario":"ana","password":"secreta"}  -> 201
POST /api/token      {"usuario":"ana","password":"secreta"}  -> 200 + token
GET  /api/perfil     Bearer <token_user>                     -> 200
GET  /api/admin/panel Bearer <token_user>                    -> 403
POST /api/token      {"usuario":"admin","password":"admin123"} -> 200 + token
GET  /api/admin/panel Bearer <token_admin>                   -> 200
GET  /api/perfil     sin token                               -> 401

RESUMEN: 7/7 casos correctos. ¡Hito 2 listo!
```

_La seguridad no es una capa al final: es una propiedad de todo el flujo._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Integrar registro, login, token, perfil y panel administrativo.
- Verificar 401, 403 y 200 en el mismo flujo.
- Demostrar que las contraseñas nunca se filtran.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Registro sin filtrar contraseñas → HTTP 201
- [ ] Login del usuario registrado → HTTP 200
- [ ] El usuario accede a su perfil → HTTP 200
- [ ] El usuario no entra al panel admin → HTTP 403
- [ ] El administrador obtiene su token → HTTP 200
- [ ] El administrador entra al panel → HTTP 200
- [ ] Sin token no hay acceso → HTTP 401

## 🧩 El reto

Integra el pipeline de seguridad completo del Bloque 2 en una API de libros con usuarios, roles y rutas protegidas.

**Requisitos del contrato**

- El administrador precargado es admin/admin123; el usuario ana se registra en el flujo.
- Registro de usuarios con contraseña hasheada (nunca en texto plano).
- Login que emite un JWT con el rol del usuario.
- Perfil protegido con Bearer para cualquier usuario autenticado.
- Panel administrativo solo para rol admin; usuario normal recibe 403.
- Sin token se responde 401 y las contraseñas nunca aparecen en las respuestas.

> **Pista conceptual:** Ordena el flujo como lo vive el cliente: registro, login, token, perfil, panel. Cada paso debe poder probarse de forma independiente.

## 📦 Entregable

Cuaderno en verde, bitácora `auditorias/auditoria_sesion20.md` y tag `v0.2-hito2` en tu repositorio.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py
"""Sesión 20 · Hito 2: registro (bcrypt) -> login (JWT con rol) -> perfil (401) -> panel admin (403)."""
import os
import secrets
from datetime import datetime, timedelta, timezone
from typing import Annotated

import bcrypt
import jwt
from fastapi import Depends, FastAPI, HTTPException, status
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, Field, field_validator

app = FastAPI(title="Hito 2 · Seguridad y RBAC · DAB-2201", version="0.2.0")

# ---------------- Configuración (secreto fuera del código) ----------------
SECRET_KEY = os.environ.get("JWT_SECRET_KEY") or secrets.token_urlsafe(48)
ALGORITMO = "HS256"
EXPIRA_MIN = 15
COSTO_BCRYPT = 12


def hashear(password: str) -> bytes:
    return bcrypt.hashpw(password.encode("utf-8"), bcrypt.gensalt(COSTO_BCRYPT))


# ---------------- "Repositorio" de usuarios: solo hashes ----------------
_usuarios: dict[str, dict] = {
    # administrador precargado (contraseña de prueba del lab, guardada con hash)
    "admin": {"hash": hashear("admin123"), "rol": "admin"},
}
_SEÑUELO = hashear("no-existe")      # para que el login tarde igual si el usuario no existe


# ---------------- DTOs ----------------
class Credenciales(BaseModel):
    usuario: str = Field(..., min_length=3, max_length=30, pattern=r"^[a-zA-Z0-9_.-]+$")
    password: str = Field(..., min_length=6)

    @field_validator("password")
    @classmethod
    def max_72_bytes(cls, v: str) -> str:
        if len(v.encode("utf-8")) > 72:
            raise ValueError("la contraseña no puede exceder 72 bytes")
        return v


class UsuarioPublico(BaseModel):
    usuario: str
    rol: str


class Token(BaseModel):
    access_token: str
    token_type: str = "bearer"
    expires_in: int


# ---------------- Seguridad: autenticación (401) y autorización (403) ----------------
bearer = HTTPBearer(auto_error=False)


def _401(detalle: str = "Token inválido o ausente") -> HTTPException:
    return HTTPException(status.HTTP_401_UNAUTHORIZED, detalle, headers={"WWW-Authenticate": "Bearer"})


def usuario_actual(cred: Annotated[HTTPAuthorizationCredentials | None, Depends(bearer)]) -> UsuarioPublico:
    if cred is None:
        raise _401()
    try:
        p = jwt.decode(cred.credentials, SECRET_KEY, algorithms=[ALGORITMO],
                       options={"require": ["exp", "sub", "rol"]})
    except jwt.InvalidTokenError:            # token mal formado o alterado -> 401, nunca 500
        raise _401()
    if p["sub"] not in _usuarios:
        raise _401()
    return UsuarioPublico(usuario=p["sub"], rol=_usuarios[p["sub"]]["rol"])


def requiere_rol(*roles: str):
    def verificar(u: Annotated[UsuarioPublico, Depends(usuario_actual)]) -> UsuarioPublico:
        if u.rol not in roles:
            raise HTTPException(status.HTTP_403_FORBIDDEN, "No tienes permiso para este recurso")
        return u
    return verificar


# ---------------- Rutas (en el orden en que las vive el cliente) ----------------
@app.post("/api/registro", status_code=status.HTTP_201_CREATED, response_model=UsuarioPublico)
def registro(cred: Credenciales) -> UsuarioPublico:
    if cred.usuario in _usuarios:
        raise HTTPException(status.HTTP_409_CONFLICT, "El usuario ya existe")
    # El rol NUNCA lo decide el cliente: todo registro nuevo es "user"
    _usuarios[cred.usuario] = {"hash": hashear(cred.password), "rol": "user"}
    return UsuarioPublico(usuario=cred.usuario, rol="user")


@app.post("/api/token", response_model=Token)
def token(cred: Credenciales) -> Token:
    datos = _usuarios.get(cred.usuario)
    valido = bcrypt.checkpw(cred.password.encode("utf-8"), datos["hash"] if datos else _SEÑUELO)
    if not datos or not valido:
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "Usuario o contraseña incorrectos")
    ahora = datetime.now(timezone.utc)
    payload = {"sub": cred.usuario, "rol": datos["rol"], "iat": ahora,
               "exp": ahora + timedelta(minutes=EXPIRA_MIN)}
    return Token(access_token=jwt.encode(payload, SECRET_KEY, algorithm=ALGORITMO),
                 expires_in=EXPIRA_MIN * 60)


@app.get("/api/perfil", response_model=UsuarioPublico)
def perfil(u: Annotated[UsuarioPublico, Depends(usuario_actual)]) -> UsuarioPublico:
    return u


@app.get("/api/admin/panel")
def panel(u: Annotated[UsuarioPublico, Depends(requiere_rol("admin"))]) -> dict:
    return {"panel": "admin", "usuarios_registrados": len(_usuarios)}


Writing app.py


## 🧪 Tu contrato C-R-E-O

| Elemento | Mi definición |
| :--- | :--- |
| **Contexto** | FastAPI + bcrypt + PyJWT (HS256); clave desde `JWT_SECRET_KEY`; dependencias para autenticación y autorización. |
| **Rol** | Arquitecto backend que integra el pipeline de seguridad completo del Bloque 2. |
| **Especificación** | `POST /api/registro` → 201 con hash bcrypt, sin `password` ni `hash` en la respuesta, rol siempre `user`; `POST /api/token` → 200 con JWT (`sub`, `rol`, `exp`); `GET /api/perfil` → 200 autenticado, 401 sin token; `GET /api/admin/panel` → 200 admin, 403 user; admin precargado admin/admin123. |
| **Output** | Solo `app.py`, organizado en configuración, repositorio, DTOs, seguridad y rutas. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'Registro sin filtrar contraseñas',
  'metodo': 'POST',
  'ruta': '/api/registro',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 201,
  'no_contiene': ['password', 'hash']},
 {'descripcion': 'Login del usuario registrado',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'guardar': {'token_user': 'access_token'}},
 {'descripcion': 'El usuario accede a su perfil',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana'}},
 {'descripcion': 'El usuario no entra al panel admin',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 403},
 {'descripcion': 'El administrador obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'admin', 'password': 'admin123'},
  'estado': 200,
  'guardar': {'token_admin': 'access_token'}},
 {'descripcion': 'El administrador entra al panel',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_admin}'},
  'estado': 200},
 {'descripcion': 'Sin token no hay acceso', 'metodo': 'GET', 'ruta': '/api/perfil', 'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 7 casos · app.py
✅ 01. Registro sin filtrar contraseñas
✅ 02. Login del usuario registrado
✅ 03. El usuario accede a su perfil
✅ 04. El usuario no entra al panel admin
✅ 05. El administrador obtiene su token
✅ 06. El administrador entra al panel
✅ 07. Sin token no hay acceso
------------------------------------------------------------------------------
RESULTADO: 7/7 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion20.md` a `auditorias/auditoria_sesion20.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
